Stage 2: Generating the positioning the GNSS using the information of stage 1 timing
Author: Brandon Engelbrecht

NOTE:
    We make use of the MeerKAT/meerkat_utils package found on GH and owned by Dr. Yi-Chao Li

In [1]:
import pickle
import sys
import time 

from satellite_RFI.src import check_satellite as cs
from satellite_RFI.src import beam_model as bm

from astropy.time import Time
import astropy.units as u
import astropy.constants as cc
from datetime import datetime
import numpy as np

In [2]:
print 'start @ ' + time.asctime(time.localtime(time.time())) +'...'

start @ Sun May 30 21:28:59 2021...


In [2]:
fname = '1551055211'   

# 1551037708   done
# 1551055211   done
# 1553966342   done
# 1554156377   done
# 1556052116   done
# 1556138397   done
# 1562857793   issue

In [3]:
"""
Location of the data given the "reCalibration" file naming convention
"""
mask_level = '6'
data_save = '/idia/projects/hi_im/brandon/'+fname+'_level'+mask_level+'_mask/'


"""
Information regarding time and freqeuncy from the "reCalibration file"
"""
katdal_info = pickle.load(open(data_save+fname+'_katdal_info.p', 'rb'))
nd_s, nd_s0, freqs, nd_s0_pos, timestamps, nd_s0_coords = [katdal_info[i] 
                                                           for i in katdal_info.keys()]

In [4]:
fname_unix = (datetime.utcfromtimestamp(
    int(fname)).strftime('%Y-%m-%d %H:%M:%S'))
print 'Observational start time:\n', fname_unix, '\n'

nd_s0_unix = (datetime.utcfromtimestamp(
    int(fname)+nd_s0[0]).strftime('%Y-%m-%d %H:%M:%S'))
print 'Noise diode off scan start time:\n', nd_s0_unix



Observational start time:
2019-02-25 00:40:11 

Noise diode off scan start time:
2019-02-25 00:53:05


In [5]:
"""
NOTE: Pointing positions of the telescope
"""
pointing =  np.array([[nd_s0_coords[0][i], nd_s0_coords[1][i]] for i 
                      in range(len(nd_s0_coords[0]))])

In [6]:
"""
Reading in the TLE data catalogues for the various observation days
"""

if fname == '1551037708' or fname == '1551055211':
    source_url = 'TLE/2019_03_07_tle/'
    
if fname == '1553966342' or fname == '1554156377':
    source_url = 'TLE/2019_03_25_tle/'
    
if fname == '1556052116' or fname == '1556138397':
    source_url = 'TLE/2019_04_22_tle/'
    
sats_type = ['gps-ops', 'glo-ops', 'galileo', 'beidou', 'irnss', 'qzs', 'sbas']   # sats_type = ['geo']

msc = cs.MeerKATsite_Satellite_Catalogue(source_url=source_url, 
                                         sats_type=sats_type, 
                                         reload=False)

msc.obs_time = nd_s0_unix
msc.obs_time_list = (nd_s0 - nd_s0[0]) * u.second
msc.get_sate_coords()

load sat catalogue from TLE/2019_03_07_tle/ gps-ops.txt
load sat catalogue from TLE/2019_03_07_tle/ glo-ops.txt
load sat catalogue from TLE/2019_03_07_tle/ galileo.txt
load sat catalogue from TLE/2019_03_07_tle/ beidou.txt
load sat catalogue from TLE/2019_03_07_tle/ irnss.txt
load sat catalogue from TLE/2019_03_07_tle/ qzs.txt
load sat catalogue from TLE/2019_03_07_tle/ sbas.txt
Time range 2019-02-25 00:53:05.000 - 2019-02-25 02:28:04.589
Satellite      gps-ops has   93 satellites  [use   9.56 s]
Satellite      glo-ops has   72 satellites  [use   7.36 s]
Satellite      galileo has   78 satellites  [use   8.03 s]
Satellite       beidou has   38 satellites  [use   8.02 s]
Satellite        irnss has    8 satellites  [use   2.29 s]
Satellite          qzs has   12 satellites  [use   3.07 s]
Satellite         sbas has   48 satellites  [use   9.89 s]


In [7]:
def satelite_timestream(frequency, pointing_position):
    '''
    Produces the angular seperation plots for the freqeuncy given and pointing
    Uses some global variabes like 
    Frequency - ini, end, channel'''
    
    ## Code from satellite_timestream
    freq = frequency  
#     beam_func = beam_model.Khans_beam_model(freq=freq)
    beam_func = bm.Cosine_beam_model(freq=freq)       # using cosine beam model

    stype, sname, stemperature = [], [], []

    for _sat_info in msc.itersats_temperature(pointings=pointing_position, beam_func=beam_func):
        _time_list, _sat_type, _sat_names, _temperature = _sat_info
        print 'Sat. Name includes %s ...'%_sat_names[0] # includes the name of each sat
        print 'N_freq x N_time x N_sats = %d x %d x %d '%_temperature.shape
        print
        np.array(stype.append(_sat_type)), sname.append(_sat_names), stemperature.append(np.sum(_temperature[:,:,:], axis=2))

        del _temperature 
    return stype, sname, stemperature


In [8]:
# Frequency channels  
channel_start_idx = 600   # @ 980MHz
channel_end_idx = 3082    # @ 1500MHz
freq_band = freqs[channel_start_idx:channel_end_idx]

In [9]:
sat_type, sat_names, stemperature = satelite_timestream(
    frequency=freq_band, pointing_position=pointing)

Sat. Name includes GPS BIIF-7  ...
N_freq x N_time x N_sats = 2482 x 2204 x 93 

Sat. Name includes COSMOS 2426 ...
N_freq x N_time x N_sats = 2482 x 2204 x 72 

Sat. Name includes GSAT0209 (PRN E09) ...
N_freq x N_time x N_sats = 2482 x 2204 x 78 

Sat. Name includes BEIDOU-3 G1 ...
N_freq x N_time x N_sats = 2482 x 2204 x 38 

Sat. Name includes IRNSS-1C ...
N_freq x N_time x N_sats = 2482 x 2204 x 8 

Sat. Name includes QZS-2  (MICHIBIKI-2) ...
N_freq x N_time x N_sats = 2482 x 2204 x 12 

Sat. Name includes GSAT-8 ...
N_freq x N_time x N_sats = 2482 x 2204 x 48 




In [10]:
# Created a dictionary and pickled values for the GPS, Galileo, GLO and Bei

sat_pos = {
    'sat_name': sat_type,
    'angular': stemperature 
}

pickle.dump(sat_pos, open(data_save+fname+"_satellite_angular_position.p", "wb"))

In [ ]:
print 'end @ ' + time.asctime(time.localtime(time.time())) +'#'

### -----------------------------------------------------END-------------------------------------------------------